In [2]:
import numpy as np
import csv
import cumbersome_model
import UNet_family
import UNet_attention
import time
import torch
import os
import random
import shutil

os.environ["CUDA_VISIBLE_DEVICES"]="0"

In [3]:
def read_train_data(file_name):
    with open(file_name, 'r', newline='') as f:
        lines = csv.reader(f)
        data = []
        for line in lines:
            data.append(line)

    data = np.array(data).astype(np.float)
    return data


def cut_data(file_name):
    with open(file_name, 'r', newline='') as f:
        lines = csv.reader(f)
        raw_data = []
        for line in lines:
            raw_data.append(line)
    raw_data = np.array(raw_data).astype(np.float)
    total = int(len(raw_data[0]) / 1024)
    for i in range(total):
        table = raw_data[:, i * 1024:(i + 1) * 1024]
        filename = './temp2/' + str(i) + '.csv'
        with open(filename, 'w', newline='') as csvfile:
            writer = csv.writer(csvfile)
            writer.writerows(table)
    return total


def glue_data(file_name, total, output):
    gluedata = 0
    for i in range(total):
        file_name1 = file_name + 'output{}.csv'.format(str(i))
        with open(file_name1, 'r', newline='') as f:
            lines = csv.reader(f)
            raw_data = []
            for line in lines:
                raw_data.append(line)
        raw_data = np.array(raw_data).astype(np.float)
        #print(i)
        if i == 0:
            gluedata = raw_data
        else:
            smooth = (gluedata[:, -1] + raw_data[:, 1]) / 2
            gluedata[:, -1] = smooth
            raw_data[:, 1] = smooth
            gluedata = np.append(gluedata, raw_data, axis=1)
    #print(gluedata.shape)
    filename2 = output
    with open(filename2, 'w', newline='') as csvfile:
        writer = csv.writer(csvfile)
        writer.writerows(gluedata)
        #print("GLUE DONE!" + filename2)


def save_data(data, filename):
    with open(filename, 'w', newline='') as csvfile:
        writer = csv.writer(csvfile)
        writer.writerows(data)

def dataDelete(path):
    try:
        shutil.rmtree(path)
    except OSError as e:
        print(e)
    else:
        pass
        #print("The directory is deleted successfully")


def decode_data(data, std_num, mode=5, trainset=0):
    Channel_location = ["FP1", "FP2",
                        "F7", "F3", "FZ", "F4", "F8",
                        "T7", "C3", "CZ", "C4", "T8",
                        "P7", "P3", "PZ", "P4", "P8",
                        "O1", "O2"]
    
    if mode == "ICUNet":
        model = cumbersome_model2.UNet1(n_channels=30, n_classes=30)
        resumeLoc = 'G:/共用雲端硬碟/CNElab_張功逸_Cary_EEGARNet/2.code/UNetpp1D/block_Result/0420_RealEEG_2_0/modelsave' + '/checkpoint.pth.tar'
    
    elif mode == "ICUNet_block":
        model = cumbersome_model2.UNet1(n_channels=30, n_classes=30)
        resumeLoc = 'G:/共用雲端硬碟/CNElab_張功逸_Cary_EEGARNet/2.code/UNetpp1D/block_Result/0420_RealEEG_2_1/modelsave' + '/checkpoint.pth.tar'
    
    elif mode == "UNetpp":
        model = UNet_family.NestedUNet3(num_classes=30)
        resumeLoc = 'G:/共用雲端硬碟/CNElab_張功逸_Cary_EEGARNet/2.code/UNetpp1D/block_Result/0420_RealEEG_3_0/modelsave' + '/checkpoint.pth.tar'
    
    elif mode == "UNetpp_block":
        model = UNet_family.NestedUNet3(num_classes=30)
        resumeLoc = 'G:/共用雲端硬碟/CNElab_張功逸_Cary_EEGARNet/2.code/UNetpp1D/block_Result/0420_RealEEG_3_1/modelsave' + '/checkpoint.pth.tar'
    
    elif mode == "Trans":
        model = UNet_attention.UNetpp3_Transformer(num_classes=30)
        resumeLoc = 'G:/共用雲端硬碟/CNElab_張功逸_Cary_EEGARNet/2.code/UNetpp1D/block_Result/0420_RealEEG_4_0/modelsave' + '/checkpoint.pth.tar'
    
    elif mode == "Trans_block":
        model = UNet_attention.UNetpp3_Transformer(num_classes=30)
        resumeLoc = 'G:/共用雲端硬碟/CNElab_張功逸_Cary_EEGARNet/2.code/UNetpp1D/block_Result/0420_RealEEG_4_1/modelsave' + '/checkpoint.pth.tar'
     
    #model = complex_cnn.Complex_CNN(in_channels=1, out_channels=1, datanum=1024, bilinear=True)
    #model = model.cuda()
    #resumeLoc = './1103_1_RealEEG_1' + '/modelsave/checkpoint.pth.tar'
    checkpoint = torch.load(resumeLoc, map_location='cpu')
    #start_epoch = checkpoint['epoch']
    #print(checkpoint)
    model.load_state_dict(checkpoint['state_dict'],False)
    model.eval()
    with torch.no_grad():
        # run the mdoel
        data = data[np.newaxis, :, :]
        data = torch.Tensor(data)
        #decode = model(data.cuda())
        #decode = model(data)
        if mode == "UNetpp" or mode == "UNetpp_block" or mode == "Trans" or mode == "Trans_block":
            decode1, decode2, decode = model(data)
        else:
            decode = model(data)
        if int(std_num) != 0:
            decode = decode * std_num
    decode = np.array(decode.cpu()).astype(np.float)
    return decode


# model = tf.keras.models.load_model('./denoise_model/')

In [3]:
def main(nname, artifact, data_num):
    #block_ch = block_num*2+1
    for j in range(1):#50
        second1 = time.time()
        #j = j+2
        name = "G:/共用雲端硬碟/CNElab_張功逸_Cary_EEGARNet/4.dataset/testing/Resting/Raw/" + str(artifact) + "/" + str(data_num) + ".csv"
        #name = "D:/resting_test/block_result/" + str(2) + "_block" + str(block_ch) + ".csv"

        # -------------------cutting_data---------------------------
        try:
            os.mkdir("./temp2/")
        except OSError as e:
            dataDelete("./temp2/")
            os.mkdir("./temp2/")
            print(e)

        #try:
        total = cut_data(name)
        #except:
        #print("jump out!")
        #continue
        # -------------------decode_data---------------------------
        for i in range(total):
            # file_name = './Real_EEG/test/Brain/510_{}.csv'.format(str(i))
            # data_clean = read_train_data(file_name)
            # torch_PSD(data_clean[0])

            file_name = './temp2/{}.csv'.format(str(i))
            data_noise = read_train_data(file_name)
            data_new = data_noise
            #print(data_new.shape)
            
            '''
            if block_ch:
                random_num = np.random.randint(30, size=block_ch)
                zero_array = np.zeros(1024)
                for k in range(block_ch):
                    data_noise[random_num[k], :] = zero_array
            '''
            
            data_clean = data_noise
            # torch_PSD(data_noise[0])

            # print(data.shape, std)
            std = np.std(data_noise)
            avg = np.average(data_noise)

            # UNet
            d_data = decode_data(data_noise, std/2, nname)#std
            d_data = d_data[0]

            outputname = "./temp2/output{}.csv".format(str(i))
            save_data(d_data, outputname)
            # save_data(d_hidden, outputname)
            # save_data(d_hidden, hiddenname)
            #print(outputname, "OK")

        # --------------------glue_data----------------------------
        #outputname = "D:/resting_test/" + nname + "/" + trainset + "/" + str(j) + ".csv"
        #outputname = "D:/resting_test/block_result/" + nname + "/" + str(block_ch) + "/" + str(j) + ".csv"
        outputname = "G:/共用雲端硬碟/CNElab_張功逸_Cary_EEGARNet/4.dataset/testing/Resting/" + nname + "/" + artifact + "/" + str(data_num) + ".csv"
        glue_data("./temp2/", total, outputname)
        # -------------------delete_data---------------------------
        dataDelete("./temp2/")
        second2 = time.time()

        print(j, "decode time: ", second2 - second1)
        # plt.savefig(str(int(j)) + "Hz_Time_domain")
        # plt.close()

In [5]:
if __name__ == '__main__':
    modelname = ["ICUNet", "ICUNet_block", "UNetpp", "UNetpp_block", "Trans", "Trans_block"]
    artifacts = ["Artifacts", "ChannelNoise", "Eye", "Heart", "LineNoise", "Muscle", "Other"]
    for i in range(6):
        i=i+5
        for j in range(7):
            #j=j+1
            for k in range(49):
                try:
                    k=k+446
                    print(modelname[i], artifacts[j], k)
                    main(modelname[i], artifacts[j], k)
                except:
                    continue

Trans_block Artifacts 446
[WinError 183] 當檔案已存在時，無法建立該檔案。: './temp2/'
0 decode time:  72.50652003288269
Trans_block Artifacts 447
0 decode time:  83.5598292350769
Trans_block Artifacts 448
0 decode time:  83.14794278144836
Trans_block Artifacts 449
0 decode time:  84.74364733695984
Trans_block Artifacts 450
0 decode time:  85.82845830917358
Trans_block Artifacts 451
0 decode time:  83.50171065330505
Trans_block Artifacts 452
0 decode time:  86.712411403656
Trans_block Artifacts 453
0 decode time:  66.62965846061707
Trans_block Artifacts 454
Trans_block Artifacts 455
[WinError 183] 當檔案已存在時，無法建立該檔案。: './temp2/'
0 decode time:  87.33571004867554
Trans_block Artifacts 456
0 decode time:  89.2958505153656
Trans_block Artifacts 457
0 decode time:  85.90647983551025
Trans_block Artifacts 458
0 decode time:  90.54277229309082
Trans_block Artifacts 459
0 decode time:  65.26075029373169
Trans_block Artifacts 460
0 decode time:  85.32873177528381
Trans_block Artifacts 461
0 decode time:  84.87291

0 decode time:  84.82811737060547
Trans_block Eye 480
0 decode time:  84.41515684127808
Trans_block Eye 481
0 decode time:  67.46890139579773
Trans_block Eye 482
0 decode time:  84.13555908203125
Trans_block Eye 483
0 decode time:  87.52441310882568
Trans_block Eye 484
0 decode time:  75.22204637527466
Trans_block Eye 485
0 decode time:  68.7959578037262
Trans_block Eye 486
Trans_block Eye 487
[WinError 183] 當檔案已存在時，無法建立該檔案。: './temp2/'
0 decode time:  84.60235476493835
Trans_block Eye 488
0 decode time:  93.72696375846863
Trans_block Eye 489
Trans_block Eye 490
[WinError 183] 當檔案已存在時，無法建立該檔案。: './temp2/'
0 decode time:  88.05085802078247
Trans_block Eye 491
0 decode time:  91.32301688194275
Trans_block Eye 492
0 decode time:  68.39872241020203
Trans_block Eye 493
0 decode time:  90.97166752815247
Trans_block Eye 494
0 decode time:  86.848801612854
Trans_block Heart 446
Trans_block Heart 447
[WinError 183] 當檔案已存在時，無法建立該檔案。: './temp2/'
Trans_block Heart 448
[WinError 183] 當檔案已存在時，無法建立該檔

0 decode time:  93.32687616348267
Trans_block Muscle 459
Trans_block Muscle 460
[WinError 183] 當檔案已存在時，無法建立該檔案。: './temp2/'
Trans_block Muscle 461
[WinError 183] 當檔案已存在時，無法建立該檔案。: './temp2/'
0 decode time:  89.60377788543701
Trans_block Muscle 462
0 decode time:  86.8047308921814
Trans_block Muscle 463
Trans_block Muscle 464
[WinError 183] 當檔案已存在時，無法建立該檔案。: './temp2/'
0 decode time:  89.1475977897644
Trans_block Muscle 465
0 decode time:  90.58106970787048
Trans_block Muscle 466
0 decode time:  87.18488025665283
Trans_block Muscle 467
0 decode time:  87.4454071521759
Trans_block Muscle 468
Trans_block Muscle 469
[WinError 183] 當檔案已存在時，無法建立該檔案。: './temp2/'
0 decode time:  96.75863695144653
Trans_block Muscle 470
0 decode time:  85.72700309753418
Trans_block Muscle 471
Trans_block Muscle 472
[WinError 183] 當檔案已存在時，無法建立該檔案。: './temp2/'
0 decode time:  88.54015040397644
Trans_block Muscle 473
0 decode time:  90.53467154502869
Trans_block Muscle 474
0 decode time:  85.53294134140015
Trans_b

In [29]:
name = "D:/resting_test/Raw/" + str(2) + ".csv"
raw_data = read_train_data(name)
print(raw_data.shape)

block_ch=9
random_num = np.random.randint(30, size=block_ch)
print(random_num)
zero_array = np.zeros(raw_data.shape[1])
for k in range(block_ch):
    raw_data[random_num[k], :] = zero_array
    
print(raw_data.shape)

filename2 = "D:/resting_test/block_result/" + str(2) + "_block" + str(block_ch) + ".csv"
with open(filename2, 'w', newline='') as csvfile:
    writer = csv.writer(csvfile)
    writer.writerows(raw_data)
    print("GLUE DONE!" + filename2)


(30, 77005)
[27  1 13  5 20 25 17  2 19]
(30, 77005)
GLUE DONE!D:/resting_test/block_result/2_block9.csv


In [19]:
root = "G:/共用雲端硬碟/CNElab_張功逸_Cary_EEGARNet/4.dataset/testing/Resting/"
model_name = "Trans_block/Channel_base/"
artifacts = ["Artifacts", "ChannelNoise", "Eye", "Heart", "LineNoise", "Muscle", "Other"]
channels = ['FP1', 'FP2', 'F7', 'F3', 'FZ', 'F4', 'F8', 'FT7', "FC3", "FCZ", "FC4", 'FT8', 'T8', 'C3', 'CZ', 'C4', 'T4', 'TP7', 'CP3', 'CPZ', 'CP4', 'TP8', 'T5', 'P3', "PZ", 'P4', 'T6', 'O1', 'OZ', 'O2']
os.mkdir(root+model_name)
for i in range(30):
    #os.mkdir(root+model_name+artifacts[i])
    os.mkdir(root+model_name+channels[i])

In [ ]:
def partial_channel2(signal1, signal2, partial_num):
    random_num = np.random.randint(30, size=partial_num)
    #print("partial_channel: ", random_num)
    zero_array = np.zeros(1024)
    for i in range(partial_num):
        #signal1[random_num[i], :] = zero_array
        signal2[random_num[i], :] = zero_array

    return signal1, signal2



In [17]:
partial = []
for i in range(39400):
    partial_num = np.random.randint(22, size=1)
    partial.append(partial_num)
    #random_num = np.random.randint(30, size=partial_num)
    #print("partial_num:", partial_num)
    #print("partial_channel: ", random_num)
partial = np.array(partial)    
with open('radomBlock_number.csv', 'w', newline='') as csvfile:
        writer = csv.writer(csvfile)
        writer.writerows(partial)

In [11]:
import numpy as np
import matplotlib.pyplot as plt


def eeg_fft(eeg_data_1, eeg_data_2):
    # Function to perform FFT and get frequency bins
    def perform_fft(signal, fs):
        n = len(signal)
        freq = np.fft.fftfreq(n, 1/fs)
        fft_values = np.fft.fft(signal)
        return freq, fft_values

    # Generate sample EEG data: 30 channels, 1024 time points, 256 Hz sampling rate
    fs = 256.0  # Sampling frequency (Hz)

    # Initialize empty arrays to store FFT differences
    fft_diff = np.zeros_like(eeg_data_1, dtype=complex)
    eeg1_fft = np.zeros_like(eeg_data_1, dtype=complex)
    eeg2_fft = np.zeros_like(eeg_data_1, dtype=complex)

    # Loop through each channel to compute FFT and find the difference
    for i in range(eeg_data_1.shape[0]):
        freq, fft_values_1 = perform_fft(eeg_data_1[i, :], fs)
        _, fft_values_2 = perform_fft(eeg_data_2[i, :], fs)

        # Compute the difference in frequency domain
        eeg1_fft[i, :] = np.abs(fft_values_1)
        eeg2_fft[i, :] = np.abs(fft_values_2)
        
    fft_diff = eeg2_fft-eeg1_fft
    return fft_diff[:,0:201], freq[0:201]

file_name1 = './Real_EEG/test/' + 'Brain' + '/' + '460_51.csv'
file_name2 = './Real_EEG/test/' + 'Eye' + '/' + '460_51.csv'


eeg_data_1 = read_train_data(file_name1) # clean
eeg_data_2 = read_train_data(file_name2) # noisy

dif_fft, freq = eeg_fft(eeg_data_1, eeg_data_2)
freq = freq.tolist()
print(freq)
save_data(freq, './fft_freq.csv')

[0.0, 0.25, 0.5, 0.75, 1.0, 1.25, 1.5, 1.75, 2.0, 2.25, 2.5, 2.75, 3.0, 3.25, 3.5, 3.75, 4.0, 4.25, 4.5, 4.75, 5.0, 5.25, 5.5, 5.75, 6.0, 6.25, 6.5, 6.75, 7.0, 7.25, 7.5, 7.75, 8.0, 8.25, 8.5, 8.75, 9.0, 9.25, 9.5, 9.75, 10.0, 10.25, 10.5, 10.75, 11.0, 11.25, 11.5, 11.75, 12.0, 12.25, 12.5, 12.75, 13.0, 13.25, 13.5, 13.75, 14.0, 14.25, 14.5, 14.75, 15.0, 15.25, 15.5, 15.75, 16.0, 16.25, 16.5, 16.75, 17.0, 17.25, 17.5, 17.75, 18.0, 18.25, 18.5, 18.75, 19.0, 19.25, 19.5, 19.75, 20.0, 20.25, 20.5, 20.75, 21.0, 21.25, 21.5, 21.75, 22.0, 22.25, 22.5, 22.75, 23.0, 23.25, 23.5, 23.75, 24.0, 24.25, 24.5, 24.75, 25.0, 25.25, 25.5, 25.75, 26.0, 26.25, 26.5, 26.75, 27.0, 27.25, 27.5, 27.75, 28.0, 28.25, 28.5, 28.75, 29.0, 29.25, 29.5, 29.75, 30.0, 30.25, 30.5, 30.75, 31.0, 31.25, 31.5, 31.75, 32.0, 32.25, 32.5, 32.75, 33.0, 33.25, 33.5, 33.75, 34.0, 34.25, 34.5, 34.75, 35.0, 35.25, 35.5, 35.75, 36.0, 36.25, 36.5, 36.75, 37.0, 37.25, 37.5, 37.75, 38.0, 38.25, 38.5, 38.75, 39.0, 39.25, 39.5, 39.75,

/tmp/ipykernel_6392/1250591231.py:8: DeprecationWarning: `np.float` is a deprecated alias for the builtin `float`. To silence this warning, use `float` by itself. Doing this will not modify any behavior and is safe. If you specifically wanted the numpy scalar type, use `np.float64` here.
Deprecated in NumPy 1.20; for more details and guidance: https://numpy.org/devdocs/release/1.20.0-notes.html#deprecations
  data = np.array(data).astype(np.float)


Error: iterable expected, not float